### Analysis of sample SRR18152477

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path


In [23]:
sample_name = 'SRR18152477'
path_qc = f'/nfs/research/goldman/anoufa/data/nicola_MNMs_paper/{sample_name}/{sample_name}_qc.tsv'

In [20]:
df_qc = pd.read_csv(path_qc, sep='\t')

# df_qc.info()

# Column Clean_depth to int
clean_depth_list = df_qc['Clean_depth'].tolist()
# Remove non int values
clean_depth_list = [x for x in clean_depth_list if x != '.']
# convert to int
clean_depth_list = list(map(int, clean_depth_list))
np.median(clean_depth_list)


np.float64(852.0)

In [24]:
# Row in result df

result_df_p = "/nfs/research/goldman/anoufa/data/MAPLE_output/processed_placements_results_masked_1.0_0.1_1_0_30000.1.tsv"

df_result = pd.read_csv(result_df_p, sep='\t')

df_sample = df_result[df_result['sample_name'] == sample_name]
df_sample

,sample_name,consensus,masked,random,n_masked_cons,n_masked_masked,prop_gen_masked,prop_dist_reduced,consensus_mutations_to_tree,masked_mutations_to_tree,consensus_placement,masked_placement,masked_mutations_masked,closest_variant,n_candidates,closeness
6180,SRR18152477,3.395864,0.0,3.772982,2878,5470,0.08668,1.0,C3276T;G4642A;C21110T,NaN,in2664694:0.12492180300950116,in2770999:1.0,C21595T;G21618C;G22813K;G22898A;G22917T;G22992...,SRR18783056,26766,11


In [22]:
# PATH MAPLE FILE

input_file = "/nfs/research/goldman/anoufa/data/nicola_MNMs_paper/SRR18152477/maple_file_seqs_del_masked.maple"
output_file = "/nfs/research/goldman/anoufa/data/nicola_MNMs_paper/SRR18152477/maple_file_seqs.fasta"

# Load the reference
with open(input_file) as f:
    lines = f.readlines()

sequences = {}
ref_name, ref_seq = None, []
current_name = None
edits = []

for line in lines:
    line = line.strip()
    if not line:
        continue

    if line.startswith(">"):
        if ref_name is None:  # First FASTA entry is reference
            ref_name = line[1:]
            continue
        else:
            if current_name:  # Save previous section
                sequences[current_name] = edits
            current_name = line[1:]
            edits = []
    else:
        if ref_seq == []:  # still reading reference
            ref_seq.append(line)
        else:
            parts = line.split()
            edits.append(parts)

# Save last one
if current_name and edits:
    sequences[current_name] = edits

ref_seq = list("".join(ref_seq))  # list so we can mutate

fasta_out = []

for name, changes in sequences.items():
    seq = ref_seq.copy()

    for change in changes:
        if len(change) == 2:  # base + position
            base, pos = change
            pos = int(pos) - 1
            if base.lower() == "n":
                seq[pos] = "N"
            elif base == "-":
                seq[pos] = ""  # delete one base
            else:
                try:
                    seq[pos] = base.upper()
                except:
                    print(f"Error processing pos {pos}, {base.upper()}")

        elif len(change) == 3:  # base, pos, length/count
            base, pos, count = change
            pos, count = int(pos) - 1, int(count)

            if base.lower() == "n":
                for i in range(count):
                    if pos + i < len(seq):
                        seq[pos + i] = "N"
            elif base == "-":  # deletion
                for i in range(count):
                    if pos + i < len(seq):
                        seq[pos + i] = ""
            else:
                # Shouldn't happen, but keep for robustness
                seq[pos] = base.upper()

    seq_str = "".join(seq)
    fasta_out.append(f">{name}\n{seq_str}")

# Write output fasta
# Write ref sequence first
Path(output_file).write_text(f">{ref_name}\n{''.join(ref_seq)}\n" + "\n".join(fasta_out))
print(f"Written {len(fasta_out)} sequences to {output_file}")


Written 3 sequences to /nfs/research/goldman/anoufa/data/nicola_MNMs_paper/SRR18152477/maple_file_seqs.fasta


In [16]:
import snipit

In [ ]:
# SNIPIT LINE

# ~/.pyenv/versions/3.11.6/bin/snipit /nfs/research/goldman/anoufa/data/nicola_MNMs_paper/SRR18152477/maple_file_seqs.fasta -t nt --output-file /nfs/research/goldman/anoufa/figs/snipit_test_fig --flip-vertical